# Lesson 6, Part 1

In [ ]:
# initializations
import numpy as np
from numpy.polynomial import Polynomial as P
import matplotlib.pyplot as plt

In the text it is proved that ln *bx* - ln *b* = ln *x* for all positive *b* and *x*.
Let's check this for various *b* and *x* using Simpson's rule with proportionate spacing.

In [ ]:
def simpson_ln(b,xmin=1,num=1000):
    num = 2 * (num//2) + 1
    x_pts = np.linspace(xmin,b*xmin,num)
    dx = (x_pts[-1] - x_pts[0]) / (num - 1)
    y_pts = 1 / x_pts
    weight = np.zeros(num)
    weight[0] = weight[-1] = 1/3
    weight[1:-1:2] = 4/3
    weight[2:-2:2] = 2/3 
    wtd_sum_y = np.sum(weight * y_pts)
    return wtd_sum_y * dx

for b in [2,4,8,16,32]:
    log_b = []
    for xmin in [0.125,0.25,0.5,1,2,4,8,16,32,64]:
        log_b.append(simpson_ln(b,xmin))
    mean_log_b = sum(log_b) / len(log_b)
    max_rel_diverge = (max(log_b) - min(log_b)) / mean_log_b
    print('Maximum relative divergence for b =',b,'is',max_rel_diverge)

It follows that ln *b*<sup>*p*</sup> = *p* ln *b* for any *p*. 
Let's check using Taylor expansions that ln 2 ~= 2 ln(sqrt 2).

In [ ]:
def sqrt_2(tol=1E-14):   # use easy sqrt method from Cb6_Sec1 workbook
    estimate = 1.5
    while True:
        diff = 2 / estimate - estimate
        estimate += 0.5 * diff
        if abs(diff) < tol:
            break
    return estimate

def taylor_ln_better(order,x):
    coef = np.zeros(order+1)
    for k in range(1,order+1,2):
        coef[k] = 2 / k 
    return P(coef)((x-1)/(x+1)) 

print('Estimated square root of 2 is',sqrt_2(),', which deviates by',sqrt_2()-np.sqrt(2),'from numpy value')
print('Deviation of ln(2) from 2*ln(sqrt(2)) using various Taylor approximations:')
for order in [3,7,11,15,19,23,27,31,35]:
    ln_2 = taylor_ln_better(order,2)
    ln_sqrt2 =  taylor_ln_better(order,sqrt_2())
    print('order',order,':',ln_2 - 2 * ln_sqrt2)
print('Last approximation for ln(2) is',ln_2,'\nwith error',ln_2-np.log(2))

Let's rewrite taylor_ln_better to compute estimates more efficiently.

In [ ]:
def taylor_ln_refined(n,x):   # n corresponds here to order//2
    v = (x - 1) / (x + 1)
    v_square = v * v
    estimate = 1 / (2 * n + 1)
    while n > 0:
        n -= 1
        estimate = 1 / (2 * n + 1) + v_square * estimate
    return 2 * v * estimate 

print('Does taylor_ln_refined(n,2) equal taylor_ln_better(2*n+1,2)?')
print('True for n between 0 and 20 except:')
for n in range(21):
    y1 = taylor_ln_refined(n,30)
    y2 = taylor_ln_better(2*n+1,30)
    if y1!=y2:
        print('n =',n,': difference =',round(y1-y2,18))

We saw in lesson Level 5 Part 1 that Python stores numbers 
in the form *mantissa* 2\*\*_exponent_,
where 0.5<=*mantissa*<1 and *exponent* is an integer. 
The code here computes their natural logarithm as 
ln(*mantissa*) + *exponent* ln 2.

In [ ]:
ln_2 = taylor_ln_refined(35,2)

def compute_ln(x,n=15):
    mantissa,exponent = np.frexp(x)
    ln_mantissa = taylor_ln_refined(n,mantissa)
    return ln_mantissa + exponent * ln_2

x = np.linspace(0.01,100,1000)
y = compute_ln(x)
plt.title('ln(x) using compute_ln')
plt.plot(x,y)
plt.figure()
plt.title('Divergence in ln(x) of Simpson versus compute_ln')
z = [simpson_ln(val,1,num=1000) for val in x]  # higher num in simpson_ln will reduce divergence
plt.plot(x,z-y)
plt.show()

How high must *n* get in compute_ln(*x,n*) before the estimate stops changing?
The highest comes for *mantissa*=0.5, since that is farthest from 1.

In [ ]:
for mantissa in np.linspace(0.5,0.95,10):
    mantissa = round(mantissa,2)
    old = 0
    for n in range(3,21):
        new = taylor_ln_refined(n,mantissa)
        if  abs(new-old) < 1E-16:
            break
        old = new
    print('mantissa =',mantissa,': n =',n)

While compute_ln is faster than simpson_ln and better handles a broad range of *x*, 
numpy offers a function **log** that calculates similarly to compute_ln and is much faster.

In [ ]:
from time import time as T

x = 300
print('ln',x,'= \nsimpson_ln',simpson_ln(x),'\ncompute_ln',compute_ln(x),'\nnp.log    ',np.log(x))
iters,tic = 1000,T()
for iter in range(iters): simpson_ln(x)
print('simpson_ln takes time',T()-tic); tic = T()
for iter in range(iters): compute_ln(x)
print('compute_ln takes time',T()-tic); tic = T()
for iter in range(iters): np.log(x)
print('np.log takes time',T()-tic)

The base *b* of a logarithm is the number for which log *b* = 1.
Let's use the Newton-Raphson algorithm to estimate the base of the natural logarithm.

In [ ]:
def inverse_ln(val,seed=1,tol=1E-14):
    x = seed
    for n in range(100):
        y = np.log(x) - val
        if abs(y) < tol:
            break
        deriv = 1 / x
        x -= y / deriv
        # x -= y * x    # combines two preceding steps
    return x

def taylor_exp(order):  # from L5_2
    inv_factorial = 1
    coef = np.ones(order+1)
    for n in range(1,order+1):
        inv_factorial /=  n
        coef[n] = inv_factorial
    return P(coef)

print('The inverse natural logarithm of 1 is',inverse_ln(1))
print('This is remarkably close to exp(1) = ',taylor_exp(20)(1))

Logarithms to other bases are constant multiples of the natural logarithm, 
where the multiple assures that log(base) = 1. The most popular of those bases are 10 and 2, 
for which numpy offers the functions **log10** and **log2**.

In [ ]:
x = np.linspace(2,20,100)
ratio = np.log10(x) / np.log(x)
mean_ratio = np.sum(ratio) / len(ratio)
estimated_ratio = np.log10(inverse_ln(1))
print('Mean ratio of log10(x) to ln(x) is',mean_ratio,'\nwhich differs from estimated ratio by',round(mean_ratio - estimated_ratio,18))
print('Maximum relative divergence in log10/ln is',np.min(ratio)/np.max(ratio)-1)
print('\nlog10 rounds to within 1 of the shift in leading digits from the decimal point')
print('x       log10 (approx)')
for z in (5E-35,3E-10,8E16,2E45):
    print(z,'  ',round(np.log10(z),1))
ratio = np.log2(x) / np.log(x)
mean_ratio = np.sum(ratio) / len(ratio)
estimated_ratio = np.log2(inverse_ln(1))
print('\nMean ratio of log2(x) to ln(x) is',mean_ratio,'\nwhich differs from estimated ratio by',round(mean_ratio - estimated_ratio,20))
print('Maximum relative divergence in log2/ln is',np.min(ratio)/np.max(ratio)-1)
print('\nlog2 rounds to within 1 of exponent in mantissa * 2**exponent ')
print('x      exponent  log2 (approx)')  
for z in (5E-35,3E-10,8E16,2E45):
    mantissa,exponent = np.frexp(z)
    print(z,'  ',exponent,'     ',round(np.log2(z),1))

The natural logarithm roughly corresponds to fractional differences for small changes, 
insofar as ln(*z/y*) ≅ (*z - y*) / *y* or (*z - y*) / *z*. A better approximation is 2(*z - y*) / (*y* + *z*).

In [ ]:
y = 100
print(' y    z   frac_vs_y  frac_vs_z  est_log_change  log_change')
for z in range(50,155,10):
    frac_vs_y = (z - y) / y
    frac_vs_z = (z - y) / z
    est_log_change = (z - y) * 2 / (y + z)
    log_change = np.log(z/y)
    print(y,'  ',z,'  ',frac_vs_y,'   ',round(frac_vs_z,3),'      ',round(est_log_change,3),'       ',round(log_change,3))

In measuring long-term growth, it is better to average the logarithmic changes than the fractional changes. 
Consider an investment that grows by fraction *x* one year and shrinks by fraction *x* the next. 
It loses a fraction *x*<sup>2</sup> of its value, w0hich the average logarithmic change will reveal, yet its average fractional change is zero.

In [ ]:
print('Total net change for investment that grows by fraction x one period and -x the next')
print(' x   net frac change   net log change   nominal difference')
for x in np.linspace(0.03,0.6,20):
    net_frac_change = round(-x**2,4)
    net_log_change = round(np.log(1-x**2),4)
    difference = round(net_frac_change - net_log_change,4)
    print(round(x,2),'    ',net_frac_change,'       ',net_log_change,'         ',difference)

Plotting in log terms is useful when values vary by several orders of magnitude. 
Matplotlib uses **semilogy** to plot *y* in log terms and **loglog** to plot both *x* and *y* in log terms.

In [ ]:
num = 1000
x = np.linspace(0.01,10,num)
y = np.zeros((num,5)) # makes array of zeros num high and 5 wide
for n in range(5):
    y[:,n] = x**-(n+1)
plt.title('Standard plot of negative powers of x')
plt.plot(x,y)     # all plots look like an L and are hard to distinguish
plt.figure()
plt.title('Semilog plot of negative powers of x')
plt.semilogy(x,y) # easy to distinguish
plt.legend(labels = (-1,-2,-3,-4,-5))
plt.figure()
plt.title('Loglog plot of negative powers of x')
plt.loglog(x,y)   # straight lines because log(y)=-n*log(x)
plt.legend(labels = (-1,-2,-3,-4,-5))
plt.show()

Here's an example that uses **semilogx** to plot *x* in log terms.

In [ ]:
plt.style.use('seaborn-v0_8')
x = np.linspace(0.1,1,1000)
logx = np.log(1+x)
approx_logx = x / (1 + 0.5 * x)
plt.title("Standard plot of ln(1+x) and x/(1+0.5x) approximation")
plt.plot(x,logx,x,approx_logx)
plt.legend(['ln(1+x)','x/(1+0.5x)'])
plt.xlabel('x')            
plt.figure()
plt.title("ln(1+x) and x/(1+0.5x) approximation with x plotted in log terms")
plt.semilogx(x,logx,x,approx_logx)
plt.legend(['ln(1+x)','2x/(2+x)'])
plt.xlabel('x')
plt.show()